## Business Data Analytics Project
This notebook performs exploratory data analysis (EDA), visualization, and predictive modeling on a synthetic business dataset. The goal is to showcase data analyst, business analyst, and program manager skills.

In [ ]:

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Set plotting style
sns.set(style="whitegrid")


In [ ]:

# Load the synthetic dataset
file_path = 'synthetic_business_data.csv'
df = pd.read_csv(file_path, parse_dates=['Date'])

# Display basic info
print('Dataset shape:', df.shape)
df.head()


In [ ]:

# Summary statistics
summary = df.describe(include='all')
display(summary)


In [ ]:

# Revenue by product category
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='Product_Category', y='Revenue', estimator=sum, ci=None)
plt.title('Total Revenue by Product Category')
plt.xticks(rotation=45)
plt.ylabel('Total Revenue')
plt.show()


In [ ]:

# Aggregate monthly revenue
df['Month_Year'] = df['Date'].dt.to_period('M')
monthly_revenue = df.groupby('Month_Year')['Revenue'].sum().reset_index()

plt.figure(figsize=(10,5))
plt.plot(monthly_revenue['Month_Year'].astype(str), monthly_revenue['Revenue'])
plt.xticks(rotation=60)
plt.title('Monthly Total Revenue Over Time')
plt.xlabel('Month-Year')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()


In [ ]:

# Correlation matrix for numerical features
num_cols = ['Units_Sold','Revenue','Cost','Profit','Marketing_Spend','Customer_Satisfaction','Month']
corr = df[num_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


In [ ]:

# Prepare features and target for regression (predict Profit)
X = df[['Units_Sold','Marketing_Spend','Customer_Satisfaction','Month','Product_Category','Region']]
y = df['Profit']

# Identify categorical and numerical columns
categorical_cols = ['Product_Category','Region']
numerical_cols = ['Units_Sold','Marketing_Spend','Customer_Satisfaction','Month']

# Preprocess: one-hot encoding for categorical variables
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Regression model (RandomForestRegressor for complexity)
model = RandomForestRegressor(random_state=42)

# Create pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

# Train model
pipeline.fit(X_train, y_train)

# Predict
preds = pipeline.predict(X_test)

# Evaluate
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print(f'Profit Prediction RMSE: {rmse:.2f}')
print(f'Profit Prediction R^2: {r2:.3f}')


In [ ]:

# Create binary target: 1 if profit above median else 0
median_profit = df['Profit'].median()
df['High_Profit'] = (df['Profit'] > median_profit).astype(int)

X_cls = df[['Units_Sold','Marketing_Spend','Customer_Satisfaction','Month','Product_Category','Region']]
y_cls = df['High_Profit']

# Split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Use preprocessing similar to regression
preprocessor_c = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

# Model
clf = RandomForestClassifier(random_state=42)

# Pipeline
pipeline_c = Pipeline([
    ('preprocessor', preprocessor_c),
    ('model', clf)
])

# Train
pipeline_c.fit(X_train_c, y_train_c)

# Predict
preds_c = pipeline_c.predict(X_test_c)

# Evaluate
acc = accuracy_score(y_test_c, preds_c)
print(f'Classification Accuracy (High vs Low Profit): {acc:.3f}')
